# OceanCleanup System 03 — Optimization Example

This notebook adapts the multi-objective optimization workflow from the lecture "System design principles" from 15.09.2026 and the example workbook: `EXAMPLE_Artificial_Reef.ipynb` to the OceanCleanup System 03 offshore plastic-collection barrier. It follows the same structure:

- Define design variables and bounds
- Specify multiple objective functions and preference mappings
- Build constraints
- Run a Genetic Algorithm (GA) using two (potentially three) aggregation paradigms
- Visualize preference functions and optimization results

## Problem

System 03 is a passive, offshore plastic-collection system: a long U-shaped floating barrier towed slowly through a garbage patch by two vessels, funneling floating debris toward a retention zone at the apex. Unlike a fixed coastal structure, its design is a trade-off between capturing as much plastic as possible and avoiding harm to marine life, all while remaining operable by a small crew at reasonable cost.

Design choices — how deep its skirt hangs below the surface, how far apart the two towing vessels hold the ends, how fast the system is towed, how fine its screen mesh is, and how many systems are deployed — all affect capture rate, bycatch risk, fishing-ground blockage, and cost. The barrier length itself is treated as fixed **(maybe this need so be adapted in the future when we want to experiment with the vessel spacing )**.

**Note:** Values in the following implementation are placeholders and not researched values.

## Importing Required Packages


In [2]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize

# Define default plotting parameters
plt.rcParams['font.size'] = '10'
plt.rcParams['savefig.dpi'] = 300

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm

ModuleNotFoundError: No module named 'genetic_algorithm_pfm'

## Design Variables and Bounds

The System 03 design is parameterized using five design variables (four continuous, one integer). They follow from the stakeholder objectives shown below: each variable influences at least one objective, and most influence several, which is what makes this a trade-off problem. The variables are numbered from left to right as in the scheme.

![Stakeholder objectives and design variables](../Our%20project/Stakeholder_Objectives_and_optimisation/stakeholder_objective_variable_scheme.png)

| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Skirt depth below waterline | m | Continuous |
| `x2` | Distance between support vessels (vessel spacing) | m | Continuous |
| `x3` | Number of systems deployed | – | Integer |
| `x4` | Towing speed | m/s | Continuous |
| `x5` | Mesh / screen size | mm | Continuous |

Bounds and values below are placeholder estimates — replace with values we can justify from project research:

- `x1` skirt depth: a deeper skirt catches more submerged plastic but adds material, drag, and contact with animals in the water column.
- `x2` vessel spacing: a wider mouth captures more plastic but needs more towing force and closes more sea area, meaning fishing area to fishers.
- `x3` number of systems: more systems remove more plastic in total, but raise the total cost, the area closed to fishers, and the ecological contact. It must be a whole number, so it is marked `'int'` in `var_type_mixed`.
- `x4` towing speed: a higher speed captures more per hour but raises drag and fuel cost, gives animals less time to avoid the barrier, and lets small fragments escape.
- `x5` mesh size: a finer mesh retains more microplastic but increases fouling and the risk of trapping small marine life.

`X_irl` holds a rough reference design of today's situation (one System 03, so `x3 = 1`) for later comparison against the optimized solutions — update it once you have better sources.

In [1]:
# Define the names of variables for later use in plotting and analysis
'''
All of those values are place holders: put in own values.
'''
design_variables = (
    ('x1', 'Skirt depth below waterline',      'm'),
    ('x2', 'Distance between support vessels', 'm'),
    ('x3', 'Number of systems deployed',       '-'),
    ('x4', 'Towing speed',                     'm/s'),
    ('x5', 'Mesh / screen size',               'mm')
)

# Fixed system parameter - NOT a design variable (use it in the objective functions where needed)
barrier_length = 2500   # m, total length of the barrier (about 2.5 km according to The Ocean Cleanup website -> 2024)

# set bounds for all variables
b1 = [2, 6]          # x1 skirt depth [m]
#Due to wave action, buoyant plastic is forced below the ocean surface; however, its vertical distribution exhibits an exponential decrease in concentration with depth
b2 = [200, 1600]     # x2 distance between vessels [m]
# If the distance is too small, the swept area becomes too narrow; if it is too wide (width of the net 2km), the system loses its functional U-shape.
b3 = [1, 10]         # x3 number of systems [-]
# fleet is capped at 10 because of the purchase cost 
b4 = [0.3, 0.75] # x4 towing speed [m/s]
# to keep the netting tensioned, while also avoiding downwelling and ensure marine life to be able to escape 
b5 = [10, 20]        # x5 mesh size [mm]
# min. mesh size because the Plankton/Neuston needs to escape 
bounds = [b1, b2, b3, b4, b5]

# type of each variable: 'real' = continuous, 'int' = whole number (used in the GA options later)
var_type_mixed = ['real', 'real', 'int', 'real', 'real']

X_irl = [4, 1200, 1, 0.5, 10] # real world values --> place holders, put in later
plot_irl = True  # Can be True or False --> plot the IRL point or not

## Objective Functions

This model optimises five objectives, each representing one stakeholder from the stakeholder analysis (see the scheme above).

| Stakeholder | Objective | Unit | Variables | Goal |
|-------------|-----------|------|-----------|-----------|
| Investors / Donors | Annual cost | M€/yr | x1, x2, x3, x4 | minimise |
| The Ocean Cleanup | Plastic removal | t/yr | x1, x2, x3, x4, x5 | maximise |
| North Pacific commercial fishers | Fishing-area interference | km² | x2, x3 | minimise |
| Marine conservation advocates | Ecological risk (bycatch) | animals/yr | x1, x2, x3, x4, x5 | minimise |
| Citizens | Future plastic fragmentation | t/yr | x2, x3, x4, x5 | minimise |

Each objective is a simple, physically motivated expression of the design variables. The physical constants are collected in **one cell below**, so they can be replaced by researched values later without touching the functions. Constants that are shared between objectives (e.g. plastic density, plastic size distribution) are defined only once, so the objectives stay consistent with each other.

**1. Cost (Investors / Donors).** Annual cost of all systems = number of systems × (fixed vessel charter + annualised barrier material + fuel). Fuel follows from the towing power, which is the drag force times the speed:

$$P_{tow} = \frac{\tfrac{1}{2}\rho_w C_D \, x_1 x_2 \, x_4^3}{\eta_{prop}}$$

The frontal area of the skirt is approximated by skirt depth × mouth width ($x_1 x_2$). Because power scales with $x_4^3$, the cost rises quickly at high towing speeds.

**2. Plastic removal (The Ocean Cleanup).** Plastic that meets the system (surface density × mouth width × speed × operating time), multiplied by three efficiency factors between 0 and 1:
- *depth factor* $1 - e^{-x_1/d_0}$: plastic concentration decreases with depth, so a deeper skirt adds less and less;
- *speed retention* $1 / \left(1 + (x_4/v_{crit})^n\right)$: above a critical speed, plastic is pushed under the skirt and escapes. Together with the linear term in $x_4$, this gives an **optimal towing speed**;
- *mesh retention* $e^{-x_5/\lambda}$: the fraction of the plastic mass that is larger than the mesh opening. It uses an exponential size distribution with characteristic size $\lambda$, so a finer mesh retains more.

**3. Fishing-area interference (Fishers).** The sea area that is closed to fishing around each system. The U is approximated by two straight arms of $L/2$, so its length in the towing direction is $h = \sqrt{(L/2)^2 - (x_2/2)^2}$. A safety buffer $b$ is added on all sides: $A = x_3 (x_2 + 2b)(h + 2b)$.

**4. Ecological risk (Marine conservation advocates).** Expected bycatch = swept water volume ($x_1 x_2 x_4 T$) × density of vulnerable animals × probability an animal cannot escape × mesh factor. The escape probability decreases with speed ($1 - e^{-x_4/v_{avoid}}$ is the probability of *not* escaping), and a finer mesh traps more small organisms ($x_{5,ref}/x_5$).

**5. Future plastic fragmentation (Citizens).** Microplastic that is created by, or passes through, the system: plastic met × fraction smaller than the mesh ($1 - e^{-x_5/\lambda}$, the complement of the mesh retention in objective 2) × a fragmentation factor that increases with towing speed ($k \, x_4^2$, from turbulence and abrasion). This objective deliberately **conflicts** with plastic removal: more systems, a wider mouth and a higher speed catch more plastic but also produce more fragments.

In [1]:
# ---------------------------------------------------------------------------------------
# Constants for the objective functions
# ALL VALUES ARE PLACEHOLDERS --> replace with researched values
# ---------------------------------------------------------------------------------------

# General operation (used in several objectives)

uptime            = 0.7                        # fraction of the year the system is actively towing [-]
T_op              = uptime * 365 * 24 * 3600   # operational time per year in seconds [s]
rho_water         = 1025                       # sea water density [kg/m3]

# Plastic in the garbage patch (shared by objectives 2 and 5 -> keeps them consistent)

plastic_density   = 5e-5    # floating plastic mass per sea surface area [kg/m2] (= 50 kg/km2)
plastic_size_char = 50      # characteristic size of the plastic size distribution, lambda [mm]

# Objective 1 - Cost

charter_cost      = 7.5e6    # fixed cost per system: two vessels + crew per year [EUR/yr]
material_cost     = 150     # barrier / skirt material cost per m2 of skirt [EUR/m2]
barrier_lifetime  = 5       # lifetime of the barrier, used to annualise material cost [yr]
drag_coeff        = 0.85     # drag coefficient of the barrier/skirt [-]
prop_efficiency   = 0.6     # propulsion efficiency of the towing vessels [-]
fuel_cost         = 0.25    # fuel cost per kWh of delivered towing energy [EUR/kWh]

# Objective 2 - Plastic removal

plastic_depth     = 5     # e-folding depth of plastic concentration in the water column, d0 [m]
v_critical        = 1.3     # towing speed above which plastic escapes under the skirt [m/s]
escape_steepness  = 4       # how sharply retention drops above v_critical, n [-]

# Objective 3 - Fishing-area interference

safety_buffer     = 500     # exclusion buffer around the system in which fishing is not allowed, b [m]

# Objective 4 - Ecological risk

animal_density    = 2.1e-3    # density of vulnerable animals in the water column [1/m3]
v_avoid           = 0.75     # characteristic speed at which animals can still avoid the barrier [m/s]
mesh_ref          = 10      # reference (finest) mesh size for the mesh factor [mm]

# Objective 5 - Future plastic fragmentation

frag_factor       = 0.01     # fraction of passing plastic that fragments at a towing speed of 1 m/s [-] 

In [4]:
# Define objective functions

def objective_function_1(x1, x2, x3, x4, x5):
    '''
    Cost function (Investors / Donors).

    :return: float of annual cost of all systems in million EUR per year.
    '''

    material = material_cost * barrier_length * x1 / barrier_lifetime            # [EUR/yr] annualised skirt material
    power    = 0.5 * rho_water * drag_coeff * x1 * x2 * x4**3 / prop_efficiency  # [W] towing power
    fuel     = power * T_op / 3.6e6 * fuel_cost                                  # [EUR/yr] J -> kWh -> EUR

    return x3 * (charter_cost + material + fuel) / 1e6


def objective_function_2(x1, x2, x3, x4, x5):
    '''
    Plastic-removal function (The Ocean Cleanup).

    :return: float of plastic removed by all systems in tonnes per year.
    '''

    encountered     = plastic_density * x2 * x4 * T_op                   # [kg/yr] plastic in front of one system
    depth_factor    = 1 - np.exp(-x1 / plastic_depth)                    # [-] share of plastic above the skirt depth
    speed_retention = 1 / (1 + (x4 / v_critical)**escape_steepness)      # [-] share not escaping under the skirt
    mesh_retention  = np.exp(-x5 / plastic_size_char)                    # [-] share of plastic mass larger than the mesh

    return x3 * encountered * depth_factor * speed_retention * mesh_retention / 1000


def objective_function_3(x1, x2, x3, x4, x5):
    '''
    Fishing-area interference function (North Pacific commercial fishers).

    :return: float of sea area closed to fishing in km2.
    '''

    u_length = np.sqrt((barrier_length / 2)**2 - (x2 / 2)**2)            # [m] length of the U in towing direction

    return x3 * (x2 + 2 * safety_buffer) * (u_length + 2 * safety_buffer) / 1e6


def objective_function_4(x1, x2, x3, x4, x5):
    '''
    Ecological-risk function (Marine conservation advocates).

    :return: float of expected bycatch in animals per year.
    '''

    swept_volume = x1 * x2 * x4 * T_op                                   # [m3/yr] water volume passed by one system
    p_no_escape  = 1 - np.exp(-x4 / v_avoid)                             # [-] probability an animal cannot avoid the barrier
    mesh_factor  = mesh_ref / x5                                         # [-] finer mesh traps more small organisms

    return x3 * animal_density * swept_volume * p_no_escape * mesh_factor


def objective_function_5(x1, x2, x3, x4, x5):
    '''
    Future plastic-fragmentation function (Citizens).

    :return: float of microplastic created/released by all systems in tonnes per year.
    '''
    
    encountered      = plastic_density * x2 * x4 * T_op                  # [kg/yr] plastic in front of one system
    passing_fraction = 1 - np.exp(-x5 / plastic_size_char)               # [-] share of plastic mass smaller than the mesh
    fragmentation    = frag_factor * x4**2                               # [-] share that fragments, grows with speed

    return x3 * encountered * passing_fraction * fragmentation / 1000


# Define the list of objectives with their corresponding names, units and stakeholders for later use in plotting and analysis
objectives = [
    (objective_function_1, "Cost",                 "M€/yr",      "Investors / Donors"),
    (objective_function_2, "Plastic Removal",      "t/yr",       "The Ocean Cleanup"),
    (objective_function_3, "Fishing Interference", "km2",        "Commercial Fishers"),
    (objective_function_4, "Ecological Risk",      "animals/yr", "Conservation Advocates"),
    (objective_function_5, "Fragmentation",        "t/yr",       "Citizens"),
]

As in the reef example, the attainable minimum and maximum of each objective is computed as a sanity check. If a range is physically unreasonable, revisit the constants above. The ranges also define the interval over which the preference curves are drawn. The value of the reference design `X_irl` is printed for comparison.

Note: `minimize` treats `x3` (number of systems) as continuous here. This is fine for finding the range, because the extremes lie at the integer bounds 1 and 10.

In [5]:
# Finding min and max for each objective using scipy's minimize function, starting from the midpoint of the bounds
objective_minmax = {}  # Dictionary to store the min and max values for each objective
midpoints = [np.mean(b) for b in bounds]

for idx, (obj_func, name, unit, stakeholder) in enumerate(objectives):
    wrapped = lambda x, sign=1: sign * obj_func(*x)

    min_val =  minimize(wrapped, x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    max_val = -minimize(lambda x: wrapped(x, sign=-1), x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    irl_val = obj_func(*X_irl)

    objective_minmax[name] = min_val, max_val
    print(f"  Objective {idx+1} {name:<21}:  min = {min_val:>12,.2f}   max = {max_val:>12,.2f}   IRL = {irl_val:>10,.2f}  {unit}")

  Objective 1 Cost                 :  min =        10.10   max =       294.56   IRL =      11.34  M€/yr
  Objective 2 Plastic Removal      :  min =        14.34   max =     8,088.88   IRL =     437.74  t/yr
  Objective 3 Fishing Interference :  min =         2.70   max =        52.58   IRL =       4.61  km2
  Objective 4 Ecological Risk      :  min =         0.17   max =     2,431.58   IRL =      37.80  animals/yr
  Objective 5 Fragmentation        :  min =         0.03   max =       727.78   IRL =       3.00  t/yr
